<a href="https://colab.research.google.com/github/amina-nasrin/Graph-Neural-Network/blob/main/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch_geometric

In [ ]:
from torch_geometric.nn import SAGEConv

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn import SAGEConv

torch.manual_seed(1)

# Load dataset
dataset = Planetoid(root="data/CiteSeer", name="CiteSeer", split='full', transform=NormalizeFeatures())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
g = dataset[0].to(device)

# Define GraphSAGE Model
class GraphSAGE(nn.Module):
    def __init__(self, in_dimension, hidden_dimension, num_classes, num_layers, aggregator):
        super(GraphSAGE, self).__init__()
        self.num_layers = num_layers
        self.layers = nn.ModuleList()

        # First layer
        self.layers.append(SAGEConv(in_dimension, hidden_dimension, aggr=aggregator))

        # Hidden layers
        for _ in range(num_layers - 2):
            self.layers.append(SAGEConv(hidden_dimension, hidden_dimension, aggr=aggregator))

        # Last layer
        self.layers.append(SAGEConv(hidden_dimension, num_classes, aggr=aggregator))

    def forward(self, g):
        h, edge_index = g.x, g.edge_index
        for layer in self.layers[:-1]:  # Apply ReLU to all layers except the last
            h = layer(h, edge_index)
            h = F.relu(h)
        h = self.layers[-1](h, edge_index)  # Last layer (no ReLU)
        return F.log_softmax(h, dim=1)

# Define training function
def train(g, model, lr, n_epoch):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_val_acc = 0
    best_test_acc = 0

    for epoch in range(1, n_epoch + 1):
        model.train()
        out = model(g)
        loss = F.cross_entropy(out[g.train_mask], g.y[g.train_mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Evaluate the model
        model.eval()
        pred = model(g).argmax(dim=1)

        train_acc = (pred[g.train_mask] == g.y[g.train_mask]).float().mean().item()
        val_acc = (pred[g.val_mask] == g.y[g.val_mask]).float().mean().item()
        test_acc = (pred[g.test_mask] == g.y[g.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_test_acc = test_acc

        if epoch % 1 == 0:
            print(f'Epoch {epoch}, Loss: {loss:.4f}, Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f}), Test Acc: {test_acc:.4f} (Best: {best_test_acc:.4f})')

# Hyperparameters
hidden_dimension = 16
num_layers = 2
aggregator = 'mean'
learning_rate = 0.01
number_epoch = 60

# Initialize model and train
model = GraphSAGE(dataset.num_node_features, hidden_dimension, dataset.num_classes, num_layers, aggregator).to(device)
train(g, model, learning_rate, number_epoch)


Epoch 1, Loss: 1.7855, Val Acc: 0.2320 (Best: 0.2320), Test Acc: 0.1810 (Best: 0.1810)
Epoch 2, Loss: 1.7756, Val Acc: 0.2320 (Best: 0.2320), Test Acc: 0.1810 (Best: 0.1810)
Epoch 3, Loss: 1.7634, Val Acc: 0.2320 (Best: 0.2320), Test Acc: 0.1810 (Best: 0.1810)
Epoch 4, Loss: 1.7486, Val Acc: 0.2320 (Best: 0.2320), Test Acc: 0.1810 (Best: 0.1810)
Epoch 5, Loss: 1.7317, Val Acc: 0.2340 (Best: 0.2340), Test Acc: 0.1820 (Best: 0.1820)
Epoch 6, Loss: 1.7141, Val Acc: 0.3440 (Best: 0.3440), Test Acc: 0.3140 (Best: 0.3140)
Epoch 7, Loss: 1.6951, Val Acc: 0.4440 (Best: 0.4440), Test Acc: 0.4300 (Best: 0.4300)
Epoch 8, Loss: 1.6745, Val Acc: 0.5080 (Best: 0.5080), Test Acc: 0.5120 (Best: 0.5120)
Epoch 9, Loss: 1.6525, Val Acc: 0.5660 (Best: 0.5660), Test Acc: 0.5670 (Best: 0.5670)
Epoch 10, Loss: 1.6292, Val Acc: 0.6100 (Best: 0.6100), Test Acc: 0.5920 (Best: 0.5920)
Epoch 11, Loss: 1.6043, Val Acc: 0.6260 (Best: 0.6260), Test Acc: 0.6160 (Best: 0.6160)
Epoch 12, Loss: 1.5780, Val Acc: 0.6380 (

In [ ]:
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch invpcid_single ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
ad